# ML signals harness

A working notebook for iterating on ML-based trading signals: register features, train a model, and evaluate it against a naive baseline. Adding a new **feature** is one line in the `FEATURES` dict below, adding a new **model architecture** is one `SkorchModel` subclass, and running a new **experiment** is one `run_experiment()`/`run_sweep()` call -- nothing else in the notebook needs to change.

Run cells top to bottom once; after that, re-run only the section you're iterating on (e.g. tweak `FEATURES` and re-run from section 1 down).

In [2]:
# Autoreload so edits to tam/ (if you're editing the package alongside this
# notebook) show up without restarting the kernel.
%load_ext autoreload
%autoreload 2

import tam

print("tam-quant OK --", tam.__file__)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
tam-quant OK -- /Users/ehudadler/AppleDev/Personal/fin/tam/__init__.py


## 0. Data + universe

`marketdata_eod` is the self-service EOD lake (`Symbol`/data-explorer), not a third-party vendor -- swap `Registry.get(DataProvider, "marketdata_eod")` for `"yfinance"` only if you need a ticker outside that lake. `repository.ingest(..., warn=False)` fills any gaps in the local Parquet cache before anything else reads from it.

In [4]:
import random
from datetime import date, timedelta

from tam.basket.matrix import price_matrix
from tam.basket.universe import PitIndexUniverse
from tam.data.providers import DataProvider
from tam.data.repository import DataRepository
from tam.data.schema import CLOSE
from tam.data.storage import DataStore
from tam.marketdata.eod_provider import MarketDataEodProvider  # noqa: F401 -- import registers "marketdata_eod"
from tam.registry import Registry

N_NAMES = 10    # how many S&P 500 names to sample -- raise once you're past quick iteration
SEED = 0        # same SEED -> same N_NAMES tickers; change it to sample a different set
HORIZON = 3     # forward-return label window, in trading days

universe = PitIndexUniverse(index="sp500")
all_tickers = universe.constituents(date.today())
TICKERS = random.Random(SEED).sample(all_tickers, min(N_NAMES, len(all_tickers)))

repository = DataRepository(
    Registry.get(DataProvider, "marketdata_eod"),
    Registry.create(DataStore, "parquet", "data/eod"),
)

end = date.today()
start = end - timedelta(days=365 * 6)

repository.ingest(TICKERS, start, end, warn=False)

closes = price_matrix(repository, TICKERS, start, end, column=CLOSE, warn=False)
TICKERS = [t for t in TICKERS if t in closes.columns]
print(f"{len(TICKERS)} tickers, {closes.shape[0]} trading days, through {closes.index.max().date()}")

10 tickers, 1506 trading days, through 2026-09-02


## 1. Features -- add one by adding a line here

`FEATURES` is the entire feature set. `Registry.names(Factor)` below prints every registered factor name (`RsiFactor`, `MacdFactor`, `TrailingReturnFactor`, `RealizedVolFactor`, `SmaDistanceFactor`, `RollingSharpe`, ...). To try a brand new feature, either register one of those under a new name/config, or define a small custom `Factor` subclass in the cell below and add it to the dict -- nothing downstream (`store.with_labels`, `run_experiment`, `run_sweep`) needs to change.

In [5]:
from tam.basket.factors import Factor, MacdFactor, RealizedVolFactor, RsiFactor, SmaDistanceFactor, TrailingReturnFactor

print("registered factors:", Registry.names(Factor))

registered factors: ['cross_sectional_rank', 'expected_shortfall', 'intraday_volatility', 'macd', 'max_drawdown', 'mean_return', 'overnight_alpha', 'overnight_beta', 'persistence', 'realized_vol', 'rsi', 'sharpe', 'sma_distance', 'trailing_return']


In [6]:
# Add/remove/reconfigure features here -- this dict IS the feature set.
FEATURES: dict[str, Factor] = {
    "ret_5d": TrailingReturnFactor(5),
    "ret_20d": TrailingReturnFactor(20),
    "rsi_14": RsiFactor(14),
    "macd": MacdFactor(),
    "vol_20d": RealizedVolFactor(20),
    "sma_dist_50": SmaDistanceFactor(50),
}


# Example of a custom feature -- a Factor takes a rolling window of closes
# and returns one score per ticker. Uncomment + add to FEATURES above to try it.
#
# class MomentumAccelFactor(Factor):
#     """Difference between two trailing-return windows -- "is momentum
#     speeding up or slowing down", not just "is it positive"."""
#
#     def __init__(self, fast: int = 5, slow: int = 20):
#         self._fast = fast
#         self._slow = slow
#
#     def compute(self, closes, as_of):
#         fast_window = self._window(closes, as_of, self._fast)
#         slow_window = self._window(closes, as_of, self._slow)
#         fast_ret = fast_window.iloc[-1] / fast_window.iloc[0] - 1
#         slow_ret = slow_window.iloc[-1] / slow_window.iloc[0] - 1
#         return fast_ret - slow_ret

In [7]:
from tam.ml.feature_store import FeatureStore

# cache_dir under data/ (already gitignored) -- keyed on (tickers, dates,
# horizon, every Factor's own config), so a cache hit skips recomputation
# entirely whenever you re-run with the same inputs.
store = FeatureStore(repository, cache_dir="data/feature_cache")
store.register_many(FEATURES)
store.feature_names

['ret_5d', 'ret_20d', 'rsi_14', 'macd', 'vol_20d', 'sma_dist_50']

In [8]:
# Sanity-check the panel before training anything on it.
preview = store.with_labels(TICKERS, start, end, horizon=HORIZON)
print(preview.shape)
preview.head()

(14422, 7)


ret_5d   ret_20d     rsi_14      macd   vol_20d  \
date       ticker                                                      
2020-11-30 TER     0.037713  0.256005  73.706193  0.001199  0.356260   
           FSLR    0.103200  0.073291  63.531976  0.009633  0.612197   
           REGN   -0.005224 -0.050648  37.665161 -0.006562  0.350220   
           UBER    0.025397  0.486381  67.764070  0.000571  0.644070   
           GPN     0.023545  0.237416  62.781538  0.004437  0.469602   

                   sma_dist_50  fwd_return_3d  
date       ticker                              
2020-11-30 TER        0.206450       0.030814  
           FSLR       0.167913      -0.048271  
           REGN      -0.086855      -0.033312  
           UBER       0.244343       0.057592  
           GPN        0.089309       0.002869

## 2. One experiment

`run_experiment()` does `store.with_labels()` -> `time_split()` -> fit ->
score the test split -> `ExperimentResult`, and `.report()` prints the
model-vs-baseline comparison + a per-feature IC leaderboard and returns a
5-panel chart (IC over time, feature correlation heatmap, score
distributions, ...).

In [9]:
from tam.ml.experiment import run_experiment

result = run_experiment(
    store,
    TICKERS,
    start,
    end,
    horizon=HORIZON,
    model="mlp",
    model_kwargs={"hidden": 32},
)
result.report()

model    -- mean IC: 0.0322  mean spread: 0.00511  hit rate: 0.4855
baseline -- mean IC: -0.0162  mean spread: -0.00611  hit rate: 0.4977
passed_gate: True  (beats 'ret_5d' on both mean IC and mean spread)

per-feature leaderboard -- every registered feature + the model's own score, ranked by mean IC:
    feature  mean_ic  mean_spread  hit_rate
sma_dist_50   0.0396       0.0026    0.5084
    vol_20d   0.0386       0.0053    0.4911
model_score   0.0322       0.0051    0.4855
    ret_20d   0.0310       0.0002    0.4864
     rsi_14   0.0231       0.0032    0.4911
     ret_5d  -0.0162      -0.0061    0.4977
       macd  -0.0282      -0.0114    0.4944


In [10]:
result.passed_gate  # beats the naive baseline feature on BOTH mean IC and mean quantile spread

True

## 3. Sweeps -- iterate by changing a grid, not rewriting a training loop

`run_sweep()` runs `run_experiment()` once per combination in the cartesian
product of the given grid axes and returns a leaderboard sorted by mean IC.
Any `run_experiment()` keyword can be a grid axis, including `tickers=` or
`store=` (a list of already-built alternatives).

In [ ]:
from tam.ml.experiment import run_sweep

leaderboard = run_sweep(
    dict(store=store, tickers=TICKERS, start=start, end=end, model="mlp"),
    horizon=[1, 2, 3, 5],
    model_kwargs=[{"hidden": 16}, {"hidden": 32}, {"hidden": 64}],
)
leaderboard

## 4. Add a new model architecture

Every architecture only needs to supply `_build_module(n_features) -> nn.Module` -- fit/predict/early-stopping/checkpointing/standardization all come from `SkorchModel`. Register it with `@Registry.register(Model, "name")` and it's usable anywhere a `model=` string is accepted above, including inside `run_sweep()`'s grid.

In [ ]:
import torch.nn as nn

from tam.ml.model import Model, SkorchModel


@Registry.register(Model, "deep_mlp")
class DeepMLPModel(SkorchModel):
    """A deeper variant of the built-in `mlp` -- one extra hidden layer."""

    def __init__(self, hidden: int = 64, **kwargs):
        super().__init__(**kwargs)
        self._hidden = hidden

    def _build_module(self, n_features: int) -> nn.Module:
        hidden = self._hidden
        return nn.Sequential(
            nn.Linear(n_features, hidden),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(hidden, hidden),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(hidden, hidden // 2),
            nn.ReLU(),
            nn.Linear(hidden // 2, 1),
        )


deep_result = run_experiment(store, TICKERS, start, end, horizon=HORIZON, model="deep_mlp", model_kwargs={"hidden": 64})
deep_result.report()

## 5. Save a winning model

Once a config's `passed_gate` is actually `True` (and ideally checked across more than one ticker set / date range), save it -- from there it's ready to be wired into a live trading strategy.

In [ ]:
if result.passed_gate:
    result.model.save("data/models/ml_signals_mlp")
    print("saved to data/models/ml_signals_mlp")
else:
    print("gate not passed -- keep iterating on FEATURES / the model grid before saving")